# 1. Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

S1_PATH = '../student_resource/dataset/train/train_source1.tsv'
S2_PATH = '../student_resource/dataset/train/train_source2.tsv'
S3_PATH = '../student_resource/dataset/train/train_source3.tsv'
GT_PATH = '../student_resource/dataset/train/train_ground_truth.tsv'


# 2. Dataset Overview
Avoiding full memory loads by chunking where possible.


In [ ]:
def overview_dataset(path, name):
    print(f"--- Overview of {name} ---")
    total_rows = 0
    mem_usage = 0
    dtypes = None
    columns = []
    for chunk in pd.read_csv(path, sep='\t', dtype=str, chunksize=500000):
        total_rows += len(chunk)
        mem_usage += chunk.memory_usage(deep=True).sum()
        if dtypes is None:
            dtypes = chunk.dtypes
            columns = chunk.columns.tolist()
            
    print(f"Rows: {total_rows:,}")
    print(f"Columns: {len(columns)} {columns}")
    print(f"Approx Memory: {mem_usage / 1024**2:.2f} MB")
    
    sample = pd.read_csv(path, sep='\t', nrows=3, dtype=str)
    display(sample)
    print("\n")

overview_dataset(S1_PATH, "Source 1")
overview_dataset(S2_PATH, "Source 2")
overview_dataset(S3_PATH, "Source 3")
overview_dataset(GT_PATH, "Ground Truth")


# 3. Data Quality
Analyzing missing values and duplicates.


In [ ]:
def check_data_quality(path, name):
    print(f"--- Data Quality for {name} ---")
    missing = {}
    empty_strings = {}
    total_rows = 0
    
    for chunk in pd.read_csv(path, sep='\t', dtype=str, chunksize=500000):
        total_rows += len(chunk)
        for col in chunk.columns:
            if col not in missing:
                missing[col] = 0
                empty_strings[col] = 0
            
            missing[col] += chunk[col].isna().sum()
            empty_strings[col] += (chunk[col].fillna('') == '').sum() - chunk[col].isna().sum()
            
    print("Missing values:")
    for col in missing:
        perc = (missing[col] / total_rows) * 100
        print(f"  {col}: {missing[col]:,} ({perc:.2f}%)")
        
    print("Empty string values (not null, but empty):")
    for col in empty_strings:
        perc = (empty_strings[col] / total_rows) * 100
        print(f"  {col}: {empty_strings[col]:,} ({perc:.2f}%)")
        
    # Check duplicates using unique chunks
    ids = pd.read_csv(path, sep='\t', usecols=[0], dtype=str).iloc[:,0]
    print(f"Duplicate entity IDs: {ids.duplicated().sum():,}")
    print("\n")

check_data_quality(S1_PATH, "Source 1")
check_data_quality(S2_PATH, "Source 2")
check_data_quality(S3_PATH, "Source 3")


# 4. Country Analysis


In [ ]:
def analyze_countries():
    countries = {}
    for name, path in [('S1', S1_PATH), ('S2', S2_PATH), ('S3', S3_PATH)]:
        df = pd.read_csv(path, sep='\t', usecols=['country'], dtype=str)
        counts = df['country'].value_counts()
        for c, count in counts.items():
            if c not in countries: countries[c] = {}
            countries[c][name] = count
            
    df_countries = pd.DataFrame(countries).T.fillna(0).astype(int)
    display(df_countries)
    
    df_countries.plot(kind='bar', figsize=(8,4))
    plt.title('Country Distribution Across Sources')
    plt.ylabel('Count')
    plt.show()

analyze_countries()


# 5. Business Name Analysis


In [ ]:
def analyze_names(path, name):
    print(f"--- Name Analysis for {name} ---")
    df = pd.read_csv(path, sep='\t', usecols=['business_name'], dtype=str).fillna('')
    names = df['business_name']
    
    lengths = names.str.len()
    print(f"Avg Name Length: {lengths.mean():.2f}")
    
    tokens = names.str.split().str.len()
    print(f"Avg Token Count: {tokens.mean():.2f}")
    
    val_counts = names.value_counts()
    print(f"Unique names: {len(val_counts):,}")
    print("Most common names:")
    print(val_counts.head(3))
    
    has_non_ascii = names.apply(lambda x: len(x) != len(x.encode('utf-8', 'ignore')))
    devanagari_regex = re.compile(r'[\u0900-\u097F]')
    has_devanagari = names.apply(lambda x: bool(devanagari_regex.search(x)))
    
    print(f"Names with Non-ASCII: {has_non_ascii.sum():,}")
    print(f"Names with Devanagari: {has_devanagari.sum():,}")
    
    if has_devanagari.sum() > 0:
        print("\nExamples of Devanagari names:")
        display(names[has_devanagari].sample(min(3, has_devanagari.sum()), random_state=42))
    print("\n")

analyze_names(S1_PATH, "Source 1")
analyze_names(S2_PATH, "Source 2")


# 6. Address Analysis


In [ ]:
def analyze_addresses(path, name):
    print(f"--- Address Analysis for {name} ---")
    df = pd.read_csv(path, sep='\t', usecols=['business_address'], dtype=str).fillna('')
    addrs = df['business_address']
    
    lengths = addrs.str.len()
    print(f"Avg Address Length: {lengths.mean():.2f}")
    
    has_digits = addrs.str.contains(r'\d')
    print(f"Addresses with digits (likely PIN/Building No): {has_digits.sum():,} ({(has_digits.sum()/len(addrs))*100:.2f}%)")
    
    has_landmark = addrs.str.contains(r'\bnear\b', case=False)
    print(f"Addresses with 'Near' landmarks: {has_landmark.sum():,}")
    print("\n")

analyze_addresses(S1_PATH, "Source 1")
analyze_addresses(S2_PATH, "Source 2")


# 7. Ground Truth Analysis


In [ ]:
gt = pd.read_csv(GT_PATH, sep='\t', dtype=str)
print(f"Total S1 Entities in GT: {len(gt):,}")

matches = gt['matched_entity_ids'].fillna('').apply(lambda x: x.split(',') if x else [])
match_counts = matches.apply(len)

singletons = (match_counts == 0).sum()
print(f"Singletons (0 matches): {singletons:,} ({(singletons/len(gt))*100:.2f}%)")
print(f"Average Matches per S1 (including singletons): {match_counts.mean():.2f}")
print(f"Median Matches: {match_counts.median()}")
print(f"Max Matches: {match_counts.max()}")

print("\nDistribution of Matches per S1:")
print(match_counts.value_counts().sort_index().head(10))


# 8. Verify S2/S3 Uniqueness


In [ ]:
all_matches = [m for sublist in matches for m in sublist]
total_matches = len(all_matches)
unique_matches = len(set(all_matches))

print(f"Total S2/S3 IDs in GT: {total_matches:,}")
print(f"Unique S2/S3 IDs in GT: {unique_matches:,}")

if total_matches == unique_matches:
    print("\n✅ DATA-DERIVED FINDING: Every S2/S3 ID occurs at most once! The bipartite one-to-one constraint holds.")
else:
    print("\n❌ Warning: S2/S3 IDs appear multiple times.")


# 9. Matched Record Inspection & 10. Hard/Easy Match Examples


In [ ]:
# Load subsets to inspect actual text pairs
np.random.seed(42)
sampled_s1_ids = gt[match_counts > 0].sample(5, random_state=42)['source1_entity_id'].values

def fetch_records(path, id_list):
    results = []
    for chunk in pd.read_csv(path, sep='\t', dtype=str, chunksize=500000):
        found = chunk[chunk['entity_id'].isin(id_list)]
        results.append(found)
    return pd.concat(results)

s1_sampled_df = fetch_records(S1_PATH, sampled_s1_ids)

s2s3_ids = []
for s1_id in sampled_s1_ids:
    m = gt[gt['source1_entity_id'] == s1_id]['matched_entity_ids'].values[0]
    if pd.notna(m) and m != '':
        s2s3_ids.extend(m.split(','))

lookup_s2s3 = pd.concat([fetch_records(S2_PATH, s2s3_ids), fetch_records(S3_PATH, s2s3_ids)]).set_index('entity_id')

for _, row in s1_sampled_df.iterrows():
    print("="*80)
    print(f"SOURCE 1: {row['business_name']} | {row['business_address']}")
    
    m = gt[gt['source1_entity_id'] == row['entity_id']]['matched_entity_ids'].values[0]
    for match_id in str(m).split(','):
        if match_id in lookup_s2s3.index:
            match_row = lookup_s2s3.loc[match_id]
            print(f"MATCH ({match_id}): {match_row['business_name']} | {match_row['business_address']}")


# 11. EDA Findings and Implications for Blocking

Based on the evidence from the actual dataset, here are our findings:

1. **Scale & Missing Data**: We are dealing with ~12.5M records. Over 3.3% of S2 and S3 addresses are completely missing. 
   - *Implication*: We cannot rely exclusively on postal/PIN blocking. Name-based or dense embedding fallback is required.
2. **Country Constraint**: The data contains exclusively US and India (France is only in the test set).
   - *Implication*: Blocking can be hard-partitioned by country to save ~60% of search space.
3. **Multilingual Presence**: Devanagari script is present natively in the data.
   - *Implication*: Do not ASCII-normalize. Multilingual sentence embeddings are essential for cross-script matching.
4. **Singletons**: ~5.6% of S1 entities have zero matches.
   - *Implication*: A dedicated singleton classifier or threshold logic is needed to preserve perfect precision on these cases.
5. **One-to-One Bipartite Uniqueness**: Every S2/S3 ID appears at most once in the ground truth.
   - *Implication*: A rigid post-processing step must enforce this constraint, assigning an S3 record to at most one S1 record.
6. **Address Noise**: Addresses are highly variable, with some using landmarks ("Near SBI ATM"), missing parts, or inverted schemas.
   - *Implication*: Address component parsing (House Number, Street, City) will yield much better signals than raw string Jaccard.

